# 근로기준법 GraphRAG Agent

---

## 학습 목표

- 한국 법령 계층 구조(법률 → 시행령 → 시행규칙, 장 → 조)를 반영한 온톨로지 설계
- Neo4j 지식그래프에 법령 데이터를 파싱·적재하는 파이프라인 구현
- 벡터 검색 / 풀텍스트 검색 / 그래프 탐색을 결합한 Hybrid 검색 구현
- LangChain `@tool` 4종과 `create_agent` 를 사용하는 법령 전문 Agent 구현

## 사전 준비

### 선수 학습
- PRJ04_W3_001 ~ PRJ04_W3_010 수료

### 필수 지식
- Neo4j Cypher 기초 (MATCH / MERGE / CREATE)
- Vector Index / Fulltext Index 개념
- RAG (Retrieval-Augmented Generation) 개념
- LangChain `@tool` 데코레이터 및 `create_agent` API

### 실행 환경
- Neo4j Desktop에 **`law`** 데이터베이스 사전 생성 필요
- `.env` 파일에 `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `OPENAI_API_KEY` 설정
- `data/markdown/` 폴더에 근로기준법 마크다운 3개 파일 존재

---

## **1. 환경 설정**

In [ ]:
# 👀 데모: 라이브러리 임포트 및 환경 변수 로드
import os
import re
import glob
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm

# Neo4j 관련
from neo4j import GraphDatabase
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship

# LangChain 관련
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain.agents import create_agent

# 환경 변수 로드
load_dotenv(override=True)

# API 키 확인
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 설정되지 않았습니다."
print("라이브러리 로드 완료!")

In [ ]:
# 👀 데모: Neo4j 연결 및 LLM / 임베딩 초기화
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = "legal"  # Neo4j Desktop 에서 미리 생성한 데이터베이스

# Neo4jGraph: 스키마 조회 없이 연결 (법령 KG 전용)
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    refresh_schema=False,
)

# 드라이버 (세션 직접 제어용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# LLM / 임베딩
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 연결 확인
ping = graph.query("RETURN 1 AS ok")
print(f"Neo4j 연결 완료! (database={NEO4J_DATABASE}, ping={ping[0]['ok']})")

---

## **2. 법령 온톨로지 설계**

한국 법령의 계층 구조와 Neo4j 노드·관계를 매핑합니다.

| 노드 | 주요 속성 | 설명 |
|------|-----------|------|
| `Law` | `id`, `name`, `full_text` | 법률 / 시행령 / 시행규칙 |
| `Chapter` | `id`, `title`, `full_text` | 장(章) 또는 절(節) |
| `Article` | `id`, `title`, `content`, `number`, `law_name`, `order`, `chapter_order`, `content_embedding` | 개별 조문 |

<br>

| 관계 | 방향 | 의미 |
|------|------|------|
| `HAS_CHAPTER` | Law → Chapter | 법률이 보유한 장 |
| `FIRST_ARTICLE` | Chapter → Article | 장의 첫 번째 조문 |
| `NEXT_ARTICLE` | Article → Article | 같은 장 내 순서 연결 |
| `HAS_DECREE` | Law(법률) → Law(시행령) | 법률-시행령 계층 |
| `HAS_RULE` | Law(시행령) → Law(시행규칙) | 시행령-시행규칙 계층 |

```mermaid
graph LR
    Law1["Law\n(법률)\n근로기준법"]
    Law2["Law\n(시행령)\n근로기준법 시행령"]
    Law3["Law\n(시행규칙)\n근로기준법 시행규칙"]
    Chapter["Chapter\n제4장 근로시간과 휴식"]
    Art1["Article\n제50조(근로시간)\ncontent_embedding: Vector"]
    Art2["Article\n제51조(탄력적 근로시간제)\ncontent_embedding: Vector"]

    Law1 -- HAS_DECREE --> Law2
    Law2 -- HAS_RULE --> Law3
    Law1 -- HAS_CHAPTER --> Chapter
    Chapter -- FIRST_ARTICLE --> Art1
    Art1 -- NEXT_ARTICLE --> Art2

    style Law1 fill:#e1f5fe,stroke:#01579b
    style Law2 fill:#e1f5fe,stroke:#01579b
    style Law3 fill:#e1f5fe,stroke:#01579b
    style Chapter fill:#fff3e0,stroke:#e65100
    style Art1 fill:#f1f8e9,stroke:#33691e
    style Art2 fill:#f1f8e9,stroke:#33691e
```

---

## **3. 마크다운 로드 & 파싱**

Docling 으로 변환된 마크다운 파일(3개)을 로드하고, 법령 계층(장 → 조 → 항 → 호)을 정규식으로 파싱합니다.

In [ ]:
# 👀 데모: 마크다운 파일 로드
def load_law_markdown_files(markdown_dir: str = "data/markdown") -> dict:
    """
    Docling 변환 마크다운 파일을 로드합니다.

    파일명 패턴: 근로기준법(법률)(제20520호)(20250223).md
    첫 번째 '(' 앞 텍스트를 law_name 으로 사용합니다.
    """
    md_path = Path(markdown_dir)
    law_contents = {}

    if not md_path.exists():
        print(f"마크다운 폴더가 없습니다: {markdown_dir}")
        return law_contents

    md_files = list(md_path.glob("*.md"))
    print(f"발견된 마크다운 파일: {len(md_files)}개")

    for md_file in md_files:
        filename = md_file.stem
        law_name = filename.split("(")[0].strip() if "(" in filename else filename

        with open(md_file, "r", encoding="utf-8") as f:
            content = f.read()

        law_contents[law_name] = content
        print(f"  - {law_name}: {len(content):,}자")

    return law_contents


law_contents = load_law_markdown_files("data/markdown")
print(f"\n총 {len(law_contents)}개 법령 문서 로드 완료!")

In [ ]:
# 👀 데모: LaborLawKGExtractor 클래스 정의 (정규식 4종: 장/조/항/호)
class LaborLawKGExtractor:
    """근로기준법 관련 문서에서 법령 계층 구조를 추출하여 KG 노드/관계를 생성합니다."""

    def __init__(self, law_contents: dict, graph: Neo4jGraph):
        self.law_contents = law_contents
        self.node_dict = {}
        self.relationships = []
        self.graph = graph

    def extract_structure(self) -> dict:
        """모든 법령에서 장/조/항/호 계층 구조를 추출합니다."""
        all_laws = {}
        for law_name, content in self.law_contents.items():
            all_laws[law_name] = self._extract_sections(content, law_name)
        return all_laws

    def _extract_sections(self, content: str, law_name: str) -> dict:
        """단일 법령 문서에서 계층 구조를 파싱합니다."""
        law_structure = {
            "name": law_name,
            "type": "law",
            "full_text": content,
            "chapters": [],
        }

        # 정규식 4종 ─────────────────────────────────────────────────────────
        title_pattern      = r"^(#+)\s+(.+)$"                     # 장/절 헤더
        article_pattern    = r"제(\d+)조\(([^)]+)\)\s+(.+)"        # 조문
        paragraph_pattern  = r"[①②③④⑤⑥⑦⑧⑨⑩]\s+(.+)"             # 항
        subparagraph_pattern = r"[-\s]*(\d+)\.\s+[\'\"]?(.+?)[\'\"]?$"  # 호
        # ─────────────────────────────────────────────────────────────────────

        lines = content.split("\n")
        current_chapter = None
        current_article = None
        current_paragraph = None

        for line in lines:
            # 장/절 매칭
            chapter_match = re.match(title_pattern, line)
            if chapter_match:
                title = chapter_match.group(2).strip()
                if "장" in title or "절" in title:
                    current_chapter = {"title": title, "type": "chapter", "full_text": title, "articles": []}
                    law_structure["chapters"].append(current_chapter)
                    current_article = None
                    current_paragraph = None
                continue

            # 조문 매칭
            article_match = re.search(article_pattern, line)
            if article_match:
                num   = article_match.group(1)
                title = article_match.group(2)
                body  = article_match.group(3)
                full_title = f"제{num}조({title})"
                current_article = {
                    "number": num, "title": full_title, "content": body,
                    "full_text": f"{full_title} {body}", "type": "article", "paragraphs": [],
                }
                if current_chapter is None:
                    current_chapter = {"title": "기본", "type": "chapter", "full_text": "기본", "articles": []}
                    law_structure["chapters"].append(current_chapter)
                current_chapter["articles"].append(current_article)
                current_chapter["full_text"] += f"\n{current_article['full_text']}"
                current_paragraph = None
                continue

            # 항 매칭
            paragraph_match = re.match(paragraph_pattern, line)
            if paragraph_match and current_article:
                para_content = paragraph_match.group(1)
                current_paragraph = {"content": para_content, "full_text": para_content, "type": "paragraph", "subparagraphs": []}
                current_article["paragraphs"].append(current_paragraph)
                current_article["full_text"] += f"\n{para_content}"
                if current_chapter:
                    current_chapter["full_text"] += f"\n{para_content}"
                continue

            # 호 매칭
            subparagraph_match = re.match(subparagraph_pattern, line)
            if subparagraph_match and current_paragraph:
                sub_num     = subparagraph_match.group(1)
                sub_content = subparagraph_match.group(2)
                sub = {"number": sub_num, "content": sub_content, "full_text": f"{sub_num}. {sub_content}", "type": "subparagraph"}
                current_paragraph["subparagraphs"].append(sub)
                current_paragraph["full_text"] += f"\n{sub['full_text']}"
                if current_article:
                    current_article["full_text"] += f"\n{sub['full_text']}"
                if current_chapter:
                    current_chapter["full_text"] += f"\n{sub['full_text']}"

        return law_structure

    def create_knowledge_graph(self) -> bool:
        """추출된 구조를 Neo4j KG 로 적재합니다."""
        all_laws = self.extract_structure()
        global_article_order = 0

        for law_name, law_structure in tqdm(all_laws.items(), desc="법령 온톨로지 구축 중"):
            law_id = f"law_{law_name}"
            law_node = Node(
                id=law_id, type="Law",
                properties={"id": law_id, "name": law_name, "full_text": law_structure["full_text"][:5000]},
            )
            self.node_dict[law_id] = law_node

            for chapter in law_structure["chapters"]:
                chapter_id = f"chapter_{law_name}_{chapter['title']}"
                chapter_node = Node(
                    id=chapter_id, type="Chapter",
                    properties={"id": chapter_id, "title": chapter["title"], "full_text": chapter["full_text"][:5000]},
                )
                self.node_dict[chapter_id] = chapter_node
                self.relationships.append(
                    Relationship(source=self.node_dict[law_id], target=self.node_dict[chapter_id], type="HAS_CHAPTER")
                )

                # ⚠️ NEXT_ARTICLE 은 같은 장 내에서만 연결 ─ 장마다 리셋
                prev_article_node = None
                chapter_article_order = 0

                for article in chapter["articles"]:
                    global_article_order += 1
                    chapter_article_order += 1
                    article_id = f"article_{law_name}_{article['title']}"
                    article_node = Node(
                        id=article_id, type="Article",
                        properties={
                            "id": article_id,
                            "title": article["title"],
                            "content": article["content"],
                            "number": article["number"],
                            "full_text": article["full_text"],
                            "chapter_title": chapter["title"],
                            "law_name": law_name,
                            "order": global_article_order,
                            "chapter_order": chapter_article_order,
                        },
                    )
                    self.node_dict[article_id] = article_node

                    if chapter_article_order == 1:
                        self.relationships.append(
                            Relationship(source=self.node_dict[chapter_id], target=self.node_dict[article_id], type="FIRST_ARTICLE")
                        )
                    if prev_article_node:
                        self.relationships.append(
                            Relationship(source=prev_article_node, target=self.node_dict[article_id], type="NEXT_ARTICLE")
                        )
                    prev_article_node = self.node_dict[article_id]

        nodes = list(self.node_dict.values())
        graph_doc = GraphDocument(nodes=nodes, relationships=self.relationships)
        self.graph.add_graph_documents([graph_doc])

        print(f"총 노드 수: {len(self.node_dict)}")
        print(f"총 관계 수: {len(self.relationships)}")
        print("법령 온톨로지 구축 완료!")
        return True


print("LaborLawKGExtractor 클래스 정의 완료!")

In [ ]:
# 👀 데모: 파싱 결과 미리보기 (장/조 카운트, 첫 조문 샘플)
extractor = LaborLawKGExtractor(law_contents, graph)
all_laws  = extractor.extract_structure()

for law_name, structure in all_laws.items():
    total_articles = sum(len(ch["articles"]) for ch in structure["chapters"])
    print(f"법령: {law_name}")
    print(f"  장 수: {len(structure['chapters'])}")
    print(f"  조문 수: {total_articles}")

    # 첫 번째 조문 샘플 출력
    for ch in structure["chapters"]:
        if ch["articles"]:
            first = ch["articles"][0]
            print(f"  첫 조문 샘플: [{ch['title']}] {first['title']} — {first['content'][:60]}...")
            break
    print()

---

## **4. 제약조건 + 지식그래프 구축**

`UNIQUE` 제약조건을 먼저 생성하면 `MERGE` 성능이 향상되고 중복 노드를 방지합니다.

In [ ]:
# 👀 데모: UNIQUE 제약조건 3개 생성 (Law / Chapter / Article)
def create_constraints(graph: Neo4jGraph):
    """법령 지식그래프용 UNIQUE 제약조건 생성"""
    constraints = [
        ("law_id_unique",     "CREATE CONSTRAINT law_id_unique     IF NOT EXISTS FOR (l:Law)     REQUIRE l.id IS UNIQUE"),
        ("chapter_id_unique", "CREATE CONSTRAINT chapter_id_unique IF NOT EXISTS FOR (c:Chapter) REQUIRE c.id IS UNIQUE"),
        ("article_id_unique", "CREATE CONSTRAINT article_id_unique IF NOT EXISTS FOR (a:Article) REQUIRE a.id IS UNIQUE"),
    ]
    for name, query in constraints:
        try:
            graph.query(query)
            print(f"  [OK] {name}")
        except Exception as e:
            print(f"  [SKIP] {name}: {e}")


print("제약조건 생성 중...")
create_constraints(graph)

print("\n현재 제약조건:")
for c in graph.query("SHOW CONSTRAINTS"):
    print(f"  - {c.get('name')}: {c.get('labelsOrTypes')} / {c.get('properties')}")

In [ ]:
# 👀 데모: Law / Chapter / Article MERGE + HAS_CHAPTER / FIRST_ARTICLE / NEXT_ARTICLE 관계 생성
extractor = LaborLawKGExtractor(law_contents, graph)
extractor.create_knowledge_graph()

In [ ]:
# 👀 데모: 법률-시행령-시행규칙 계층 관계 생성 (HAS_DECREE / HAS_RULE)
def create_law_hierarchy_relationships(graph: Neo4jGraph):
    """HAS_DECREE (법률→시행령), HAS_RULE (시행령→시행규칙) 관계 생성"""

    result1 = graph.query("""
        MATCH (law:Law), (decree:Law)
        WHERE law.name CONTAINS '근로기준법'
          AND NOT law.name CONTAINS '시행령'
          AND NOT law.name CONTAINS '시행규칙'
          AND decree.name CONTAINS '시행령'
        MERGE (law)-[r:HAS_DECREE]->(decree)
        RETURN count(r) AS created
    """)

    result2 = graph.query("""
        MATCH (decree:Law), (rule:Law)
        WHERE decree.name CONTAINS '시행령'
          AND rule.name CONTAINS '시행규칙'
        MERGE (decree)-[r:HAS_RULE]->(rule)
        RETURN count(r) AS created
    """)

    print(f"법률→시행령 (HAS_DECREE): {result1[0]['created']}개")
    print(f"시행령→시행규칙 (HAS_RULE): {result2[0]['created']}개")


create_law_hierarchy_relationships(graph)

# 노드 카운트 검증
print("\n[노드 카운트 검증]")
stats = graph.query("""
    MATCH (n)
    WHERE n:Law OR n:Chapter OR n:Article
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY count DESC
""")
for s in stats:
    print(f"  {s['label']}: {s['count']}개")

print("\n[법령 계층 구조]")
hierarchy = graph.query("""
    MATCH (law:Law)
    WHERE NOT law.name CONTAINS '시행령' AND NOT law.name CONTAINS '시행규칙'
    OPTIONAL MATCH (law)-[:HAS_DECREE]->(decree:Law)
    OPTIONAL MATCH (decree)-[:HAS_RULE]->(rule:Law)
    RETURN law.name AS 법률, decree.name AS 시행령, rule.name AS 시행규칙
""")
for h in hierarchy:
    print(f"  {h['법률']} → {h['시행령'] or '없음'} → {h['시행규칙'] or '없음'}")

---

## **5. 인덱스 생성 + 임베딩 적재**

Nori 풀텍스트 인덱스와 cosine 벡터 인덱스를 생성하고, OpenAI 임베딩을 Article 노드에 저장합니다.

In [ ]:
# 👀 데모: CJK 풀텍스트 인덱스 생성 (law_article_fulltext)
graph.query("""
    CREATE FULLTEXT INDEX law_article_fulltext IF NOT EXISTS
    FOR (a:Article) ON EACH [a.title, a.content]
    OPTIONS {
        indexConfig: {
            `fulltext.analyzer`: 'cjk',
            `fulltext.eventually_consistent`: true
        }
    }
""")
print("풀텍스트 인덱스 생성 완료! (CJK 분석기)")

# 벡터 인덱스 생성 (law_article_embeddings, dim=1536, cosine)
graph.query("""
    CREATE VECTOR INDEX law_article_embeddings IF NOT EXISTS
    FOR (n:Article)
    ON n.content_embedding
    OPTIONS {
        indexConfig: {
            `vector.dimensions`: 1536,
            `vector.similarity_function`: 'cosine'
        }
    }
""")
print("벡터 인덱스 생성 완료! (1536-dim, cosine)")

# 인덱스 목록 확인
print("\n현재 인덱스:")
for idx in graph.query("SHOW INDEXES"):
    if "law" in idx.get("name", "").lower() or "article" in idx.get("name", "").lower():
        print(f"  - {idx['name']} ({idx['type']})")

In [ ]:
# 👀 데모: Article 노드 임베딩 생성 → db.create.setNodeVectorProperty 로 적재 (tqdm)
articles = graph.query("""
    MATCH (a:Article)
    WHERE a.content IS NOT NULL
    RETURN a.title AS title, a.content AS content, a.law_name AS law_name
""")

print(f"임베딩 생성 대상: {len(articles)}개 조문")

success_count = 0
for article in tqdm(articles, desc="임베딩 생성 중"):
    content_text = f"{article['title']}\n\n{article['content']}"
    if content_text.strip():
        try:
            embedding = embeddings.embed_query(content_text)
            graph.query("""
                MATCH (a:Article {title: $title, law_name: $law_name})
                CALL db.create.setNodeVectorProperty(a, 'content_embedding', $embedding)
                RETURN a
            """, {"title": article["title"], "law_name": article["law_name"], "embedding": embedding})
            success_count += 1
        except Exception as e:
            print(f"  오류 ({article['title']}): {e}")

print(f"\n임베딩 적재 완료! ({success_count}/{len(articles)}개 성공)")

# 임베딩 완료 검증
check = graph.query("""
    MATCH (a:Article)
    WHERE a.content_embedding IS NOT NULL
    RETURN count(a) AS embedded_count
""")
print(f"임베딩 저장된 조문: {check[0]['embedded_count']}개")

---

## **6. 벡터 검색**

`db.index.vector.queryNodes` 를 사용하여 질의와 의미적으로 유사한 조문을 검색합니다.

In [ ]:
# 👀 데모: 벡터 검색 함수 정의
def vector_search(query: str, k: int = 3) -> list:
    """벡터 유사도 기반 조문 검색 (law_article_embeddings 인덱스 사용)"""
    query_embedding = embeddings.embed_query(query)

    with driver.session(database=NEO4J_DATABASE) as session:
        results = session.run("""
            CALL db.index.vector.queryNodes('law_article_embeddings', $k, $emb)
            YIELD node, score
            RETURN
                node.id            AS id,
                node.title         AS title,
                node.content       AS content,
                node.chapter_title AS chapter,
                node.law_name      AS law_name,
                score,
                'vector'           AS search_type
            ORDER BY score DESC
        """, {"k": k, "emb": query_embedding}).data()

    return results


print("vector_search 함수 정의 완료!")

In [ ]:
# 👀 데모: 벡터 검색 시연
test_query = "연차휴가 부여 기준"
v_results  = vector_search(test_query, k=3)

print(f"벡터 검색: '{test_query}'")
print("-" * 60)
for r in v_results:
    print(f"  [{r['score']:.4f}] {r['law_name']} / {r['chapter']} / {r['title']}")
    print(f"          {r['content'][:80]}...")

---

## **7. 풀텍스트 검색**

`db.index.fulltext.queryNodes` 를 사용하여 CJK 분석기 기반 키워드 검색을 수행합니다.

In [ ]:
# 👀 데모: 풀텍스트 검색 함수 정의
def fulltext_search(query: str, k: int = 3) -> list:
    """CJK 키워드 기반 조문 검색 (law_article_fulltext 인덱스 사용)"""
    results = graph.query("""
        CALL db.index.fulltext.queryNodes('law_article_fulltext', $query)
        YIELD node, score
        RETURN
            node.id            AS id,
            node.title         AS title,
            node.content       AS content,
            node.chapter_title AS chapter,
            node.law_name      AS law_name,
            score,
            'fulltext'         AS search_type
        ORDER BY score DESC
        LIMIT $k
    """, {"query": query, "k": k})
    return results


print("fulltext_search 함수 정의 완료!")

In [ ]:
# 👀 데모: 풀텍스트 검색 시연
ft_results = fulltext_search("근로시간 휴게", k=3)

print("풀텍스트 검색: '근로시간 휴게'")
print("-" * 60)
for r in ft_results:
    print(f"  [{r['score']:.4f}] {r['law_name']} / {r['chapter']} / {r['title']}")
    print(f"          {r['content'][:80]}...")

---

## **8. 그래프 탐색**

`NEXT_ARTICLE` / `FIRST_ARTICLE` 관계를 따라 1~2 hop 인접 조문을 확장 검색합니다.

In [ ]:
# 👀 데모: 그래프 탐색 함수 정의 (가변 경로 1..2 hops)
def graph_search(doc_ids: list) -> list:
    """
    시드 조문(doc_ids)에서 1~2 hop 인접 조문을 그래프 탐색합니다.
    NEXT_ARTICLE / FIRST_ARTICLE 관계를 모두 활용합니다.
    """
    results = graph.query("""
        MATCH (article:Article)
        WHERE article.id IN $doc_ids OR article.title IN $doc_ids
        MATCH path = (article)-[r*1..2]-(related:Article)
        WHERE article <> related
        WITH article, related, size(r) AS path_length,
             [rel IN r | type(rel)] AS relationship_types
        RETURN
            article.id    AS source_id,
            article.title AS source_title,
            COLLECT(DISTINCT {
                id:            related.id,
                title:         related.title,
                content:       related.content,
                chapter:       related.chapter_title,
                law_name:      related.law_name,
                path_length:   path_length,
                relationships: relationship_types
            }) AS related_articles
    """, {"doc_ids": doc_ids})
    return results


print("graph_search 함수 정의 완료!")

In [ ]:
# 👀 데모: 그래프 탐색 시연 (벡터 검색 결과를 시드로 사용)
seed_ids   = [r["id"] for r in v_results if r.get("id")]
g_results  = graph_search(seed_ids)

print(f"그래프 탐색: 시드 조문 {len(seed_ids)}개 → 관련 조문 확장")
print("-" * 60)
for r in g_results:
    print(f"  소스: {r['source_title']}")
    print(f"  관련 조문 수: {len(r['related_articles'])}개")
    for rel in r["related_articles"][:2]:
        print(f"    - {rel['law_name']} / {rel['chapter']} / {rel['title']} ({' → '.join(rel['relationships'])})")

---

## **9. Hybrid 결합**

벡터 + 풀텍스트 검색 결과를 dedup 병합하고, 그래프 탐색으로 관련 조문을 확장합니다.

In [ ]:
# 👀 데모: hybrid_law_search 헬퍼 함수 정의 (vector + fulltext dedup → graph 확장)
def hybrid_law_search(query: str, k: int = 3) -> dict:
    """
    Vector + Fulltext + Graph 통합 검색.

    1. 벡터 검색 (k건)
    2. 풀텍스트 검색 (k건)
    3. 중복 제거 후 ID 수집
    4. 그래프 탐색으로 인접 조문 확장
    """
    vec_results  = vector_search(query, k)
    ft_results   = fulltext_search(query, k)

    # dedup
    seen_titles  = set()
    combined     = []
    for r in vec_results:
        if r["title"] not in seen_titles:
            seen_titles.add(r["title"])
            combined.append({**r, "source": "vector"})
    for r in ft_results:
        if r["title"] not in seen_titles:
            seen_titles.add(r["title"])
            combined.append({**r, "source": "fulltext"})

    doc_ids     = [r["id"] for r in combined if r.get("id")]
    graph_results = graph_search(doc_ids) if doc_ids else []

    return {"vector": vec_results, "fulltext": ft_results, "graph": graph_results, "combined": combined}


print("hybrid_law_search 함수 정의 완료!")

In [ ]:
# 👀 데모: Hybrid 검색 시연
hybrid_results = hybrid_law_search("법정 근로시간과 연장 근로의 제한", k=3)

print("Hybrid 검색: '법정 근로시간과 연장 근로의 제한'")
print(f"  벡터:    {len(hybrid_results['vector'])}건")
print(f"  풀텍스트: {len(hybrid_results['fulltext'])}건")
print(f"  병합(dedup): {len(hybrid_results['combined'])}건")
print(f"  그래프 확장: {len(hybrid_results['graph'])}개 시드")
print()
for r in hybrid_results["combined"]:
    print(f"  [{r['source'].upper():8s}] {r['law_name']} / {r['title']}")

---

## **10. 법령 도구 4종 정의**

Agent 가 호출할 `@tool` 을 4개 정의합니다. 10K / 기업 관련 도구는 포함하지 않습니다.

| 도구 | 용도 |
|------|------|
| `search_law_by_name` | 법률명으로 Law 노드와 장 목록 조회 |
| `search_article_content` | 키워드로 조문 직접 검색 |
| `semantic_legal_search` | 자연어 질의로 의미 검색 |
| `get_related_articles` | 특정 조문의 인접 관련 조문 반환 |

In [ ]:
# 👀 데모: 법령 도구 4개 정의
@tool
def search_law_by_name(law_name: str) -> str:
    """
    법률명으로 법령을 검색합니다. 법령의 개요와 장 목록을 반환합니다.

    Args:
        law_name: 법률명 키워드 (예: 근로기준법, 시행령)
    """
    results = graph.query("""
        MATCH (l:Law)
        WHERE l.name CONTAINS $name
        OPTIONAL MATCH (l)-[:HAS_CHAPTER]->(ch:Chapter)
        RETURN l.name AS name, l.full_text AS full_text,
               collect(DISTINCT ch.title) AS chapters
        LIMIT 1
    """, {"name": law_name})

    if not results:
        return f"'{law_name}' 관련 법률을 찾을 수 없습니다."

    r = results[0]
    full_text = (r["full_text"] or "")[:500]
    return f"법률: {r['name']}\n장: {', '.join(r['chapters'])}\n내용:\n{full_text}..."


@tool
def search_article_content(keyword: str) -> str:
    """
    조문 제목 또는 내용에서 키워드를 검색합니다.

    Args:
        keyword: 검색 키워드 (예: 근로시간, 연차, 휴게)
    """
    results = graph.query("""
        MATCH (a:Article)
        WHERE a.content CONTAINS $keyword OR a.title CONTAINS $keyword
        RETURN a.title AS title, a.content AS content,
               a.chapter_title AS chapter, a.law_name AS law_name,
               a.order AS order
        ORDER BY a.order
        LIMIT 5
    """, {"keyword": keyword})

    if not results:
        return f"'{keyword}' 관련 조문을 찾을 수 없습니다."

    output = f"'{keyword}' 조문 검색 결과:\n"
    for r in results:
        content = (r["content"] or "")[:200]
        output += f"\n[{r['law_name']} / {r['chapter']} / {r['title']}]\n{content}...\n"
    return output


@tool
def semantic_legal_search(query: str, top_k: int = 3) -> str:
    """
    자연어 질의로 법령 조문을 의미 기반 검색합니다.

    Args:
        query: 자연어 질문 (예: 1주 근로시간 한도)
        top_k: 반환할 결과 수 (기본 3)
    """
    query_embedding = embeddings.embed_query(query)

    with driver.session(database=NEO4J_DATABASE) as session:
        results = session.run("""
            CALL db.index.vector.queryNodes('law_article_embeddings', $top_k, $emb)
            YIELD node, score
            RETURN node.title         AS title,
                   node.content       AS content,
                   node.chapter_title AS chapter,
                   node.law_name      AS law_name,
                   score
            ORDER BY score DESC
        """, {"top_k": top_k, "emb": query_embedding}).data()

    if not results:
        return "관련 조문을 찾을 수 없습니다."

    output = f"'{query}' 의미 검색 결과:\n"
    for r in results:
        content = (r["content"] or "")[:150]
        output += f"\n[{r['law_name']} / {r['chapter']} / {r['title']}] (유사도: {r['score']:.3f})\n{content}...\n"
    return output


@tool
def get_related_articles(article_title: str) -> str:
    """
    특정 조문과 인접하거나 관련된 다른 조문을 반환합니다.
    NEXT_ARTICLE / FIRST_ARTICLE 관계를 활용합니다.

    Args:
        article_title: 조문 제목 또는 조문 번호 (예: 제50조, 제54조)
    """
    results = graph.query("""
        MATCH (a:Article)
        WHERE a.title CONTAINS $title
        OPTIONAL MATCH (a)-[r]-(related:Article)
        RETURN a.title    AS source,
               a.law_name AS source_law,
               collect(DISTINCT {title: related.title, law_name: related.law_name, relation: type(r)}) AS related
        LIMIT 1
    """, {"title": article_title})

    if not results or not results[0].get("related"):
        return f"'{article_title}'의 관련 조문을 찾을 수 없습니다."

    r = results[0]
    output = f"{r['source']} ({r['source_law']}) 관련 조문:\n"
    for rel in r["related"]:
        if rel["title"]:
            output += f"  - {rel['title']} ({rel['law_name']}, {rel['relation']})\n"
    return output


legal_tools = [search_law_by_name, search_article_content, semantic_legal_search, get_related_articles]
print("법령 도구 4종 정의 완료!")
for t in legal_tools:
    print(f"  - {t.name}")

---

## **11. LangChain Agent 구성**

In [ ]:
# 👀 데모: LangChain 1.0 create_agent 로 법령 전문 Agent 생성
system_prompt = """당신은 한국 노동법 전문 자문 AI 입니다.
근로기준법, 근로기준법 시행령, 근로기준법 시행규칙 관련 질문에 정확하게 답변합니다.

## 보유 도구
- search_law_by_name: 법률명으로 법령 개요 및 장 목록 조회
- search_article_content: 특정 키워드가 포함된 조문 검색
- semantic_legal_search: 자연어 질의로 의미 기반 조문 검색
- get_related_articles: 특정 조문의 인접·관련 조문 반환

## 도구 사용 가이드
1. 특정 조문 번호(예: 제50조)가 언급되면 search_article_content 또는 get_related_articles 를 우선 사용합니다.
2. 개념 설명이나 의미 검색은 semantic_legal_search 를 사용합니다.
3. 시행령·시행규칙까지 범위를 넓혀야 하는 경우 여러 도구를 조합합니다.
4. 답변에 반드시 법령 출처(예: 근로기준법 제50조)를 명시합니다.
5. 법령에서 근거를 찾지 못한 경우 솔직하게 모른다고 답합니다."""

agent = create_agent(
    model=llm,
    tools=legal_tools,
    system_prompt=system_prompt,
)

print("법령 전문 Agent 생성 완료! (LangChain 1.0 create_agent)")

---

## **12. 에이전트 시연**

4가지 질의 유형으로 Agent 동작을 시연합니다.

| # | 질의 | 예상 도구 |
|---|------|----------|
| 1 | 근로기준법 제50조 1주 근로시간 한도 | search_article_content |
| 2 | 휴게시간 관련 조항 | semantic_legal_search |
| 3 | 제54조와 관련된 다른 조문 | get_related_articles |
| 4 | 연장근로 시행령·시행규칙 포함 종합 | 복합 도구 |

In [ ]:
# 👀 데모: run_agent 헬퍼 정의 (마지막 AIMessage + 도구 호출 로그)
from langchain_core.messages import AIMessage, ToolMessage


def run_agent(question: str) -> str:
    """
    Agent 를 실행하고 최종 답변과 도구 호출 로그를 출력합니다.
    """
    print(f"\n{'='*70}")
    print(f"질문: {question}")
    print("=" * 70)

    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # 도구 호출 로그
    tool_calls = []
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls.append(tc["name"])

    if tool_calls:
        print(f"[도구 호출] {' → '.join(tool_calls)}")

    # 마지막 AIMessage 추출
    final_answer = ""
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and not msg.tool_calls:
            final_answer = msg.content
            break

    print(f"\n[답변]\n{final_answer}")
    return final_answer


print("run_agent 헬퍼 정의 완료!")

In [ ]:
# 👀 데모: 질의 1 — 특정 조문 번호 검색 (search_article_content 예상)
run_agent("근로기준법 제50조에서 정한 1주 근로시간 한도는 몇 시간인가요?")

In [ ]:
# 👀 데모: 질의 2 — 의미 기반 검색 (semantic_legal_search 예상)
run_agent("휴게시간과 관련된 조항을 찾아줘")

In [ ]:
# 👀 데모: 질의 3 — 그래프 관계 탐색 (get_related_articles 예상)
run_agent("제54조와 관련된 다른 조문이 있나요?")

In [ ]:
# 👀 데모: 질의 4 — 다중 도구 조합 (법률 + 시행령 + 시행규칙)
run_agent("연장근로 관련 근로기준법 조문과 시행령, 시행규칙 내용까지 함께 정리해줘")

---

## **13. 정리**

### 핵심 요약

| 단계 | 핵심 구현 | 포인트 |
|------|-----------|--------|
| 법령 온톨로지 | `LaborLawKGExtractor` (정규식 4종) | NEXT_ARTICLE 은 **같은 장 내**에서만 연결 |
| KG 적재 | UNIQUE 제약 3개 → MERGE | HAS_DECREE / HAS_RULE 로 법령 계층 연결 |
| 인덱스 | CJK fulltext + cosine vector (dim=1536) | `db.create.setNodeVectorProperty` 로 임베딩 저장 |
| 3종 검색 | vector / fulltext / graph 각각 독립 구현 | graph 는 1~2 hop 가변 경로 |
| Hybrid | dedup 병합 → graph 확장 | 재현율(recall) 향상 |
| Agent 도구 | `@tool` 4종 (법령 전용) | Agent 가 질의 의도에 따라 도구를 자동 라우팅 |

### 핵심 Cypher 패턴

```cypher
-- 벡터 검색
CALL db.index.vector.queryNodes('law_article_embeddings', $k, $emb) YIELD node, score

-- 풀텍스트 검색
CALL db.index.fulltext.queryNodes('law_article_fulltext', $query) YIELD node, score

-- 그래프 탐색
MATCH (a:Article)-[r*1..2]-(related:Article) WHERE a <> related
```

### 다음 단계

1. **다중 법령 도메인 확장** — 최저임금법, 산업안전보건법 등 추가 법령 통합
2. **신구 법령 비교** — 법령 개정 이력을 버전 노드로 관리하여 조문 변경 추적
3. **Langfuse 모니터링** — Agent 도구 호출 비용·지연·품질을 대시보드로 관찰
4. **Re-ranking** — 도구 반환 결과를 LLM 으로 재순위화하여 정확도 향상